### Improving grpo for reinforcement learning--->  rlvr

#### downloading previous chapter notebook

In [1]:
from pathlib import Path
import requests


def download_from_github(rel_path, out=None):

    github_raw_base = (
        "https://raw.githubusercontent.com/"
        "shaheennabi/open-posttraining-system/main/"
    )

    rel_path = Path(rel_path)

    out = Path(out) if out is not None else Path(rel_path.name)

    if out.exists():
        print(f"{out} already exists")
        return out

    r = requests.get(
        github_raw_base + rel_path.as_posix()
    )
    r.raise_for_status()

    out.write_bytes(r.content)

    print(f"Downloaded {out}")

    return out

In [2]:
download_from_github(
    "training_reasoning_models_with_reinforcement_learning/rlvr_grpo_training_with_no_kl.py"
)

rlvr_grpo_training_with_no_kl.py already exists


PosixPath('rlvr_grpo_training_with_no_kl.py')

In [3]:
### rlvr_grpo training run
!uv run rlvr_grpo_training_with_no_kl.py --steps 500 --max_new_tokens 1024

cpython-3.14.5-linux-x86_64-gnu (download) ------------------------------ 34.26 MiB/34.29 MiB                                                                   Using CPython 3.14.5
Creating virtual environment at: /teamspace/studios/this_studio/open-posttraining-system/.venv
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
⠙ Preparing packages... (0/55)
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
⠙ Preparing packages... (0/55)
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
⠙ Preparing packages... (0/55)
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
⠙ Preparing packages... (0/55)
jinja2               ------------------------------     0 B/131.74 KiB
   Building open-posttraining-system @ file:///teamspace/studios/this_studio/ope
⠙ Preparing packages... (0/55)
jinja2      

In [3]:
## log file (download from previous chapter)
download_from_github(
    "training_reasoning_models_with_reinforcement_learning/train_rlvr_grpo_metrics.csv"
)

train_rlvr_grpo_metrics.csv already exists


PosixPath('train_rlvr_grpo_metrics.csv')

In [4]:
import pandas as pd

df = pd.read_csv(
    "train_rlvr_grpo_metrics.csv",
    header=None,
    names=[
        "step",
        "eval_freq",
        "loss",
        "reward_avg",
        "avg_response_len",
    ],
)

print(df.head())

   step  eval_freq      loss  reward_avg  avg_response_len
0     1         50 -1.239144        0.25            287.75
1     2         50 -3.203884        0.25            295.00
2     3         50 -0.000000        1.00            152.25
3     4         50 -0.000000        1.00            129.00
4     5         50 -5.945770        0.75            143.50


#### Inspecting the grpo training run (from sebastian raschka's notebook)

In [5]:
download_from_github(
    "evaluating_reasoning_models/eval_math_500.py"
)

eval_math_500.py already exists


PosixPath('eval_math_500.py')

In [ ]:
!uv run eval_math_500.py --dataset_size 50 --step 50 --checkpoint_path "checkpoints/qwen3-0.6B-rlvr-grpo-step00050.safetensors"

* merge csv's  

In [ ]:
import pandas as pd

train_df = pd.read_csv(
    "train_rlvr_grpo_metrics.csv",
    header=None,
    names=[
        "step",
        "eval_freq",
        "loss",
        "reward_avg",
        "avg_response_len",
    ],
)

eval_df = pd.read_csv("eval_metrics.csv")

merged_df = train_df.merge(
    eval_df[
        [
            "step",
            "eval_acc",
            "num_correct",
            "num_examples",
        ]
    ],
    on="step",
    how="left",
)

merged_df.to_csv(
    "merged_grpo_metrics.csv",
    index=False,
)

print(merged_df.head())
print(f"\nSaved: merged_grpo_metrics.csv")

In [ ]:
import csv
import matplotlib.pyplot as plt


def moving_average(values, window_fraction=0.25):
    # Smooth a noisy training signal to reveal longer-term trends during training
    window_size = max(1, int(window_fraction * len(values)))
    smoothed = []

    for i in range(len(values)):
        start_idx = max(0, i - window_size + 1)
        window_mean = sum(values[start_idx : i + 1]) / (i - start_idx + 1)
        smoothed.append(window_mean)

    return smoothed


def plot_grpo_metrics(csv_path, columns, save_as=None):
    data = {name: {"steps": [], "values": []} for name in columns}

    # Open and read CSV log file
    with Path(csv_path).open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row or not row.get("step"):
                continue

            # Use the training step as the shared x-axis across all metrics
            step = int(row["step"])

            for name in columns:
                value_str = row.get(name)
                if value_str:
                    data[name]["steps"].append(step)
                    data[name]["values"].append(float(value_str))

    # Create a fixed grid so loss, rewards, response length, etc. can be shown side by side
    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(6, 4))
    axes = axes.ravel()

    for i, name in enumerate(columns):
        steps = data[name]["steps"]
        values = data[name]["values"]

        # Skip metrics that are not present
        if not values:
            fig.delaxes(axes[i])
            continue

        # Evaluation accuracy as barplot because we don't have data for each step
        if name == "eval_acc":
            axes[i].bar(steps, values, width=20)
        else:
            axes[i].plot(steps, values, alpha=0.4)
            axes[i].plot(steps, moving_average(values))

        axes[i].set_ylabel(name)

    for j in (2, 3):
        if axes[j] in fig.axes:
            axes[j].set_xlabel("Step")

    plt.tight_layout()
    if save_as is not None:
        plt.savefig(save_as)
    plt.show()


# Plot the GRPO training run
plot_grpo_metrics(
    "train_rlvr_grpo_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    # save_as="4.pdf"
)

### Tracking more advanced Grpo performance metrics

* Advantage tracking

In [ ]:
import torch

def compute_advantage_stats(rewards_list):
    # This is what we already compute in GRPO:
    rewards = torch.tensor(rewards_list)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # These are the new statistics we add:
    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

* Entropy tracking